In [1]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
WINDOW = 223
STEP = 20
NUM_STREAMS = 500

In [3]:
current_dir = Path.cwd()
data_dir = current_dir.parent.parent / "data"
gesture_data_dir = data_dir / "gesture"
division_data_dir = data_dir / "division"
window_data_dir = division_data_dir / "data"

os.makedirs(window_data_dir, exist_ok=True)

# Loading the dataset

In [4]:
gestures = []

for file in os.listdir(gesture_data_dir):
    if not file.endswith(".csv"):
        continue

    path = os.path.join(gesture_data_dir, file)
    df = pd.read_csv(path)

    gesture = df.values.astype(np.float32)

    if len(gesture) < 20:
        continue

    gestures.append({"name": file, "data": gesture})

print(f"Loaded {len(gestures)} gestures")

Loaded 228 gestures


# Generating noise chunks that with be placed in the beginning and at the end of each stream

In [5]:
noise_chunks = []

for gesture in gestures:
    gesture = pd.DataFrame(gesture["data"])
    gesture = gesture.values.astype(np.float32)

    if len(gesture) < 20:
        continue

    tail = random.randint(
        10,
        min(15, len(gesture) // 2)
    )

    noise = gesture[-tail:]

    noise_chunks.append(noise)

print(f"Generated {len(noise_chunks)} noise chunks")
        

Generated 228 noise chunks


# Helper noise function

In [6]:
def get_random_noise():
    """
    Random concatenation of 1-3 noise chunks.
    """
    n = random.randint(1, 3)

    selected = random.choices(noise_chunks, k=n)

    return np.vstack(selected)

# Streams generation

In [7]:
targets = [] # Used to train the model to predict the start and end of gestures in a window
stream_ranges = [] # Used to split train and test data based on streams
stream_gestures = [] # Used to map gestures to streams and their positions

window_id = 0

for stream_id in range(NUM_STREAMS):
    first_window_id = window_id

    stream_parts = []
    boundaries = []

    current_pos = 0

    # Initial noise
    noise = get_random_noise()

    if len(noise):
        stream_parts.append(noise)
        current_pos += len(noise)

    # Number of random gestures
    n_gestures = random.randint(3, 4)

    selected = random.choices(gestures, k=n_gestures)

    gesture_order = 0

    for gesture_info in selected:
        gesture_name = gesture_info["name"]
        gesture = gesture_info["data"]

        # Initial noise before the gesture
        # noise = get_random_noise()

        if len(noise):
            stream_parts.append(noise)
            current_pos += len(noise)

        gesture_length = len(gesture)

        # tail = random.randint(10, min(15, max(10, gesture_length // 3)))

        active_part = gesture[:]
        # idle_tail = gesture[-tail:]

        start_idx = current_pos

        end_idx = current_pos + len(active_part) - 1

        boundaries.append((start_idx, end_idx))

        stream_gestures.append(
            {
                "stream_id": stream_id,
                "gesture_order": gesture_order,
                "gesture": gesture_name,
                "start_sample": start_idx,
                "end_sample": end_idx,
            }
        )

        gesture_order += 1

        # Active part of the gesture
        if len(active_part):
            stream_parts.append(active_part)
            current_pos += len(active_part)

        # Tail of the gesture as noise
        # if len(idle_tail):
        #     stream_parts.append(idle_tail)
        #     current_pos += len(idle_tail)

    # Final noise
    noise = get_random_noise()

    if len(noise):
        stream_parts.append(noise)
        current_pos += len(noise)

    if not stream_parts:
        continue

    stream = np.vstack(stream_parts)

    # Sliding windows
    
    if len(stream) < WINDOW:
        continue

    for left in range(0, len(stream) - WINDOW + 1, STEP):
        right = left + WINDOW

        if right > len(stream):
            x = stream[-right:]
            left = len(stream) - WINDOW
        else:
            x = stream[left:right]
        
        local_start = -1
        local_end = -1

        # Search for the first gesture start that falls within the window
        for g_start, _ in boundaries:
            if left <= g_start < right:
                local_start = g_start - left
                break

        # Search for the first gesture end that falls within the window
        for _, g_end in boundaries:
            if left <= g_end < right:
                local_end = g_end - left
                break

        # Save the window data to a CSV file
        pd.DataFrame(x).to_csv(
            os.path.join(window_data_dir, f"{window_id:06d}.csv"), index=False
        )

        # Form target information for the window
        has_start = int(local_start >= 0)
        has_end = int(local_end >= 0)

        # Normalization within [0.0, 1.0].
        # If no event, set to -1.0
        start_norm = local_start / (WINDOW - 1) if has_start else -1.0
        end_norm = local_end / (WINDOW - 1) if has_end else -1.0

        targets.append(
            {
                "window_id": window_id,
                "has_start": has_start,
                "start_idx": local_start,
                "start_norm": start_norm,
                "has_end": has_end,
                "end_idx": local_end,
                "end_norm": end_norm,
            }
        )

        window_id += 1

    last_window_id = window_id - 1

    stream_ranges.append(
        {
            "stream_id": stream_id,
            "stream_length": len(stream),
            "first_window_id": first_window_id,
            "last_window_id": last_window_id,
        }
    )

# Save metadata

pd.DataFrame(targets).to_csv(os.path.join(division_data_dir, "targets.csv"), index=False)

pd.DataFrame(stream_ranges).to_csv(
    os.path.join(division_data_dir, "stream_ranges.csv"), index=False
)

pd.DataFrame(stream_gestures).to_csv(
    os.path.join(division_data_dir, "stream_gestures.csv"), index=False
)

print(f"Generated {window_id} windows")
print(f"Targets saved to {division_data_dir}/targets.csv")
print(f"Stream ranges saved to {division_data_dir}/stream_ranges.csv")
print(f"Stream gestures saved to {division_data_dir}/stream_gestures.csv")

Generated 17329 windows
Targets saved to /Users/dmytrok/Documents/Універ/6 семестр/term6/smart-glove-ml/data/division/targets.csv
Stream ranges saved to /Users/dmytrok/Documents/Універ/6 семестр/term6/smart-glove-ml/data/division/stream_ranges.csv
Stream gestures saved to /Users/dmytrok/Documents/Універ/6 семестр/term6/smart-glove-ml/data/division/stream_gestures.csv
